<a href="https://colab.research.google.com/github/Aaidt/3D_graphics/blob/main/coding_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install openai requests matplotlib -q

In [ ]:
import os
import json
import re
import time
import requests
import subprocess
from openai import OpenAI
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
FREE_MODEL = "openrouter/free"
TOOL_MODEL = "openrouter/free"

In [ ]:
import os
import subprocess
from typing import Dict, Any, List, Tuple
import json
import time
import matplotlib.pyplot as plt

# --- Tool Definitions ---

def read_files(file_path: str) -> str:
    """Reads the content of a file."""
    try:
        with open(file_path, 'r') as f:
            content = f.read()
        return f"Content of {file_path}:\n{content}"
    except FileNotFoundError:
        return f"Error: File not found at {file_path}"
    except Exception as e:
        return f"Error reading file {file_path}: {e}"

def write_files(file_path: str, content: str) -> str:
    """Writes content to a file. Overwrites if file exists."""
    try:
        with open(file_path, 'w') as f:
            f.write(content)
        return f"Successfully wrote to {file_path}"
    except Exception as e:
        return f"Error writing to file {file_path}: {e}"

def list_files(directory_path: str = '.') -> str:
    """Lists files and directories in the specified path."""
    try:
        entries = os.listdir(directory_path)
        files = [e for e in entries if os.path.isfile(os.path.join(directory_path, e))]
        dirs = [e for e in entries if os.path.isdir(os.path.join(directory_path, e))]
        return f"Files in {directory_path}:\n{os.linesep.join(files)}\nDirectories in {directory_path}:\n{os.linesep.join(dirs)}"
    except FileNotFoundError:
        return f"Error: Directory not found at {directory_path}"
    except Exception as e:
        return f"Error listing directory {directory_path}: {e}"

def execute_python(code: str) -> str:
    """Executes Python code in a subprocess and returns stdout/stderr."""
    try:
        with open("temp_script.py", "w") as f:
            f.write(code)
        result = subprocess.run(["python", "temp_script.py"], capture_output=True, text=True, check=True)
        os.remove("temp_script.py")
        stdout, stderr = result.stdout.strip(), result.stderr.strip()
        output = ""
        if stdout: output += f"STDOUT:\n{stdout}\n"
        if stderr: output += f"STDERR:\n{stderr}\n"
        return output if output else "Execution successful."
    except subprocess.CalledProcessError as e:
        if os.path.exists("temp_script.py"): os.remove("temp_script.py")
        return f"Error: {e.stderr}"
    except Exception as e:
        return f"Error: {e}"

# --- Agent Core Logic (REAct Loop) ---

class CodingAgent:
    def __init__(self, client, model_name: str):
        self.client = client
        self.model_name = model_name
        self.history = []
        self.token_log = []
        self.tools = {
            "read_files": read_files,
            "write_files": write_files,
            "list_files": list_files,
            "execute_python": execute_python,
        }

    def _format_tools(self) -> str:
        return "\n".join([f"- {name}: {func.__doc__}" for name, func in self.tools.items()])

    def run(self, task: str, max_steps: int = 10):
        self.history = [{"role": "system", "content": f"Follow REAct pattern: THOUGHT, ACTION (JSON), OBSERVATION. Available tools:\n{self._format_tools()}"}]
        current_prompt = task

        for step in range(max_steps):
            print(f"\n--- Step {step + 1} ---")
            response_text = self._get_model_response(current_prompt)
            print(f"Agent:\n{response_text}")

            if "FINAL ANSWER:" in response_text:
                return response_text.split("FINAL ANSWER:")[-1].strip()

            try:
                action_match = json.loads(response_text.split("ACTION:")[1].split("OBSERVATION:")[0].strip())
                observation = self.tools[action_match['tool']](**action_match['args'])
                print(f"Observation: {observation}")
                current_prompt = f"OBSERVATION: {observation}"
                self.history.append({"role": "user", "content": current_prompt})
            except Exception as e:
                error_msg = f"Error parsing or executing action: {e}"
                print(error_msg)
                current_prompt = f"OBSERVATION: {error_msg}"
                self.history.append({"role": "user", "content": current_prompt})
            time.sleep(1)

    def _get_model_response(self, user_content: str) -> str:
        self.history.append({"role": "user", "content": user_content})
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=self.history
        )
        usage = response.usage
        self.token_log.append({
            "input_tokens": usage.prompt_tokens,
            "output_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens
        })
        content = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": content})
        return content

    def visualize_token_usage(self):
        if not self.token_log: return
        steps = range(1, len(self.token_log) + 1)
        plt.figure(figsize=(10, 5))
        plt.plot(steps, [x['total_tokens'] for x in self.token_log], marker='o', label='Total Tokens')
        plt.title("Token Usage over Steps")
        plt.xlabel("Step")
        plt.ylabel("Tokens")
        plt.grid(True)
        plt.show()

    def get_token_usage_summary(self):
        return {"total": sum(x['total_tokens'] for x in self.token_log), "steps": len(self.token_log)}

In [ ]:
# Initialize agent with OpenRouter client defined in previous cell
agent = CodingAgent(client=client, model_name=TOOL_MODEL)

print("--- Running Multi-step Task ---")
task = "Create a file 'hello.py' that prints 'Hello from OpenRouter'. Execute it, then list files to verify."
result = agent.run(task)
print(f"Final Result: {result}")

agent.visualize_token_usage()
print(agent.get_token_usage_summary())